# F-GAMR — Exploración guiada por lógica difusa y selección difusa del frente de Pareto para el enrutamiento multi-objetivo dinámico en SDN

**Proyecto aplicado final · IPD434 Seminario de Soft Computing · USM · 2026-2**  
**Autor:** [NOMBRE COMPLETO] · **Rol USM:** [ROL USM] · **Repositorio:** [URL DEL REPOSITORIO]

**Artículo base:** H.-A. Tran, C.-S. Duong, T.-D. Bui, V. Tong, H. T. T. Binh, *"GAMR: Revolutionizing Multi-Objective Routing in SDN Networks With Dynamic Genetic Algorithms"*, IEEE TETCI, vol. 9, n.º 4, pp. 3147–3161, 2025, doi:10.1109/TETCI.2025.3543836.

---

### Idea en una línea
GAMR explora el espacio de rutas **a ciegas** (caminatas aleatorias y mutaciones uniformes) y elige la ruta a instalar con un criterio **topológico** (menor solapamiento) que ignora la QoS que percibe el usuario. **F-GAMR** agrega dos sistemas difusos Mamdani interpretables, sin entrenamiento:

| Componente | Qué hace | Debilidad de GAMR que ataca |
|---|---|---|
| **FIS-1 — deseabilidad QoS** | Puntúa cada movimiento u→v según *progreso hacia el servidor, retardo, pérdida y carga medida* y guía con una ruleta difusa la caminata RSBI y la mutación por inserción | Exploración ciega a la QoS (debilidad 4) |
| **FIS-2 — selector de compromiso** | Elige del frente de Pareto la solución con mejor compromiso difuso retardo–pérdida | Elección final ingenua (debilidad 2) |

Todo lo demás (memoria PRSM, α = 0.5, cruce multi-punto, selección NSGA-II, parada por hipervolumen) es **idéntico** a GAMR, de modo que cualquier diferencia se atribuye a los dos sistemas difusos (con estudio de ablación).

### Cómo usar este notebook
1. Abrir con el directorio de trabajo en la **raíz del repositorio** (donde está este archivo) e instalar `requirements.txt`.
2. En la Sección 2 elegir `MODO`:
   * `'completo'` → **carga** los resultados de las 30 semillas ya ejecutadas (`results/raw_logs/`). Si no existen, los ejecuta (≈ 1.5 h con 7 procesos).
   * `'rapido'` → prueba de humo de ≈ 5 min (resultados en `results/_rapido/`, sin validez estadística).
3. **Run All.** El mismo pipeline se ejecuta sin notebook con `python run_reproduction.py`.

### Mapa del repositorio
```
F-GAMR.ipynb            ← este notebook (narrativa + resultados)
run_reproduction.py     ← script maestro de 1 comando (experimentos → tablas → figuras)
data/topologies.py      ← Oxford y Chinanet (Internet Topology Zoo, .gml incluidos)
src/network_model.py    ← plano de datos SDN, Ec. 1–6 y 14, QoS medida
src/gamr_original.py    ← GAMR fiel al paper (Sub-Alg. 1–6, Ec. 12–13)
src/baselines.py        ← Min-Hop, QoS-Dijkstra, LCP-WAC* (proxy) y NSGA-II
src/fuzzy_logic.py      ← mini-librería Mamdani (sin dependencias externas)
src/proposed_method.py  ← F-GAMR: FIS-1 + FIS-2 + modos de ablación
src/metrics_stats.py    ← HV exacto, IGD, Wilcoxon, tablas LaTeX
src/experiments.py      ← protocolo experimental paralelo (E1–E3)
src/reporting.py, src/plots.py ← tablas y figuras del informe
scripts/calibracion.py  ← calibración en semillas de validación (500–503)
notebooks/GAMR_replicacion_seminario.ipynb ← anexo: réplica de GAMR del seminario
results/                ← raw_logs/, tables/, figures/, calibracion/, replicacion_seminario/
tests/test_core.py      ← pruebas unitarias (pytest)
```

## 1. Selección de la propuesta (Fase 0 del instructivo)

Se evaluaron las tres extensiones propuestas con la matriz de decisión del instructivo (puntaje 1–10):

| Criterio (peso) | F-GAMR (difusa) | CM-GAMR (clustering + memético) | Neuro-GAMR (MLP) |
|---|---|---|---|
| Sinergia con IPD434 y contribución clara (30 %) | **9** — GA + lógica difusa, el par clásico de Soft Computing | 7 — ML no supervisado + memético | 8 — neuro-evolutivo |
| Superioridad estadística demostrable (30 %) | **8** | 8 | 6 — requiere datos de entrenamiento |
| Elegancia y ajuste a 5 páginas (20 %) | **9** — 2 FIS, superficies de control | 6 — muchas piezas | 7 |
| Rapidez y reproducibilidad (20 %) | **9** — sin entrenamiento, sin dependencias | 6 | 6 — entrenamiento y pesos |
| **Total ponderado** | **8.7** | 7.0 | 6.8 |

**Ganador: F-GAMR.** Se integra además la dominancia con restricciones de Deb para ℬₖ ≥ Bₖ (en todos los evolutivos, porque el paper no especifica su manejo).

### Del diseño inicial al diseño final: lo que dijo la evidencia
El instructivo proponía un **FIS-1 controlador** que adaptara α (memoria) y pₘ (mutación) según la volatilidad de la red, y un **FIS-2** con 4 criterios + volatilidad. Antes de los experimentos finales se calibró todo en **semillas de validación (500–503), disjuntas de las de evaluación (0–29)** — `scripts/calibracion.py`, Sección 7:

1. **El controlador adaptativo de α/pₘ empeoró a GAMR** (HV hasta −19 %): subir α en red estable colapsa la diversidad (todas las rutas comparten el tramo memorizado) y subir pₘ solo agrega desvíos (la mutación por inserción siempre alarga la ruta). Tampoco aportaron inmigrantes aleatorios. **Se descartó.**
2. El análisis de los frentes mostró que la verdadera palanca era **la calidad de la exploración**: RSBI y la mutación eligen vecinos al azar. Un FIS que puntúa la deseabilidad QoS de cada movimiento mejoró el HV en los 4 escenarios de validación → **nuevo FIS-1**.
3. En FIS-2, las entradas de solapamiento, holgura y volatilidad **no mejoraron la QoS realizada** (el retardo planificado predice el realizado con r ≈ 0.92–1.0) → base **parsimoniosa de 6 reglas** sobre retardo y pérdida.

Este recorrido es parte del análisis crítico: la hipótesis inicial fue refutada con datos y el método se rediseñó **sin tocar las semillas de evaluación**.

## 2. Configuración

In [ ]:
MODO = 'completo'     # 'completo' (carga/ejecuta 30 semillas) | 'rapido' (prueba de humo, ~5 min)
N_JOBS = None         # procesos en paralelo (None = núcleos − 1)
RECALCULAR = False    # True fuerza a re-ejecutar aunque existan resultados guardados

import sys, time, random
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

RAIZ = Path.cwd()
assert (RAIZ / 'src').exists(), 'Abre el notebook con el directorio de trabajo en la raíz del repositorio.'
sys.path.insert(0, str(RAIZ))

from data.topologies import cargar_topologia, resumen_topologias
from src.network_model import RedSDN, Evaluador, evaluar_qos, solapamiento, TABLA_II, MODELO
from src.gamr_original import GAMR, PARAM_GAMR
from src.baselines import NSGA2, MinHop, QoSDijkstra, LCPWACProxy
from src.proposed_method import (FGAMR, FIS1, FIS2, KAPPA, construir_guia, atributos_frente, aptitud_difusa,
                                 elegir_fis2)
from src.metrics_stats import metricas_conjuntas
from src.experiments import configuraciones, cargar_o_ejecutar, ORDEN, SELECTORES
from src.reporting import generar_reporte
from src import plots

plots.estilo()
plt.rcParams['figure.dpi'] = 110
BASE = RAIZ / 'results' if MODO == 'completo' else RAIZ / 'results' / '_rapido'
DIR_CRUDOS, DIR_TABLAS, DIR_FIGS = BASE / 'raw_logs', BASE / 'tables', BASE / 'figures'
DIR_NB = RAIZ / 'results' / 'figures' / 'notebook'          # figuras ilustrativas de este notebook
pd.set_option('display.width', 200, 'display.max_columns', 30)
print('Parámetros de GAMR (Tabla III):', PARAM_GAMR)
print('Supuestos del modelo de red:', MODELO)

## 3. Punto de partida: la réplica de GAMR (seminario)

En el seminario se replicó GAMR desde cero en Python puro (`notebooks/GAMR_replicacion_seminario.ipynb`). Hallazgos que **motivan** este proyecto:

* La memoria PRSM **acelera** (−11 % a −18 % de tiempo vs NSGA-II, Wilcoxon p < 10⁻⁵ en Oxford) pero **no mejora significativamente el frente** (HV/IGD con p > 0.79).
* Bajo alta carga GAMR reduce el retardo frente a Min-Hop (−15.8 % en Oxford, Nₛ = 2, Nᵣ = 100), pero un competidor consciente de la carga (LCP-WAC*) queda muy cerca.
* La Ec. 4 literal (mínimo global de utilización) es casi constante → se usa la versión del código oficial (cuello de botella por ruta).
* El paper y el código oficial difieren (el *solver* desplegado no cruza, no usa *crowding*, N = 50, Gᵐᵃˣ = 50).

**Topologías reales** (Internet Topology Zoo; el paper reporta 19 y 44 nodos):

In [ ]:
display(resumen_topologias())
resumen_sem = RAIZ / 'results' / 'replicacion_seminario' / 'resumen_resultados.md'
if resumen_sem.exists():
    display(Markdown(resumen_sem.read_text(encoding='utf-8')))
img = RAIZ / 'results' / 'replicacion_seminario' / 'fig_exp1_mapa_utilizacion.png'
if img.exists():
    display(Image(filename=str(img), width=900))

## 4. Debilidades de GAMR y qué ataca F-GAMR

| # | Debilidad (evidencia) | ¿F-GAMR la ataca? |
|---|---|---|
| 1 | α, pₘ fijos y ciegos al entorno | Se intentó (FIS adaptativo) y **se refutó** en validación → no se incluye |
| 2 | Elección final por menor solapamiento, ignora retardo/pérdida | **Sí — FIS-2** |
| 3 | PRSM de 1 ruta por servidor y BFS sin pesos | No (trabajo futuro) |
| 4 | RSBI y mutación eligen vecinos uniformemente, ciegos a la QoS | **Sí — FIS-1** |

## 5. Método propuesto: F-GAMR

```
                    estado medido Δᵗ (monitoreo LLDP: delay, loss, bw, carga)
                                     │
                 ┌───────────────────┴──────────────────┐
                 ▼                                      │
   FIS-1: w(u→v | S) = Mamdani(progreso, retardo, pérdida, carga)
                 │  ruleta  P(v) ∝ w^κ + ε
   ┌─────────────┼──────────────────────────────────┐   │
   │  Inicialización híbrida (α = 0.5)              │   │
   │    PSBI (memoria PRSM + BFS)  │  RSBI GUIADA ◄──┘   │
   │  Cruce multi-punto en Ω  →  Mutación GUIADA         │
   │  Eliminación de ciclos → Selección NSGA-II          │
   │  Parada ε(G) = 0 (HV estancado 10 generaciones)     │
   └──────────────────────────┬──────────────────────────┘
                              ▼ frente de Pareto F¹
         FIS-2: aptitud = Mamdani(retardo, pérdida)  →  argmax  →  reglas OpenFlow
                              ▼
                     actualización de la memoria PRSM
```

### 5.1 Lógica difusa (Mamdani)
Cada sistema usa funciones de pertenencia triangulares/trapezoidales, conjunción por **mínimo**, implicación por **mínimo** (recorte), agregación por **máximo** y defuzzificación por **centroide**:
$$y^* = \frac{\int y\,\mu_{\text{agr}}(y)\,dy}{\int \mu_{\text{agr}}(y)\,dy},\qquad \mu_{\text{agr}}(y)=\max_r \min\big(\tau_r,\ \mu_{B_r}(y)\big),\qquad \tau_r=\min_{i\in r}\mu_{A_{r,i}}(x_i)$$
Implementación propia y vectorizada en `src/fuzzy_logic.py` (sin `scikit-fuzzy`, para que la reproducción no dependa de una librería poco mantenida).

### 5.2 FIS-1: deseabilidad QoS de movimientos (exploración guiada)
Para un movimiento $u\to v$ dentro de una ruta hacia el servidor $S$, con el estado **medido** del intervalo:
$$\pi_{uv}=\tfrac12\big(h_S(u)-h_S(v)+1\big)\in\{0,\tfrac12,1\},\quad d_e=\tfrac{delay_e}{100\,\text{ms}},\quad l_e=\tfrac{loss_e}{10\,\%},\quad c_e=\tfrac{bg_e^t}{bw_e^t}$$
donde $h_S$ son los saltos al servidor (aleja / mismo nivel / acerca). FIS-1 entrega $w_{uv}\in[0,1]$ y la caminata RSBI elige el siguiente nodo con
$$P(v\mid u)=\frac{w_{uv}^{\kappa}+\varepsilon}{\sum_{v'\in\mathcal{N}(u)\setminus\text{visitados}} \big(w_{uv'}^{\kappa}+\varepsilon\big)},\qquad \kappa=2,\ \varepsilon=0.01$$
conservando el *backtracking* de RSBI. Con $\varepsilon>0$ **ningún movimiento válido queda prohibido** → se mantiene la exploración (no es un Dijkstra disfrazado). La mutación elige $\tilde\lambda\in\gamma$ con $P\propto q(\lambda_1,\tilde\lambda)\,q(\tilde\lambda,\lambda_2)$, donde $q$ es la deseabilidad con progreso neutro.

In [ ]:
print(f'FIS-1: {len(FIS1.reglas)} reglas (κ = {KAPPA})')
for i, r in enumerate(FIS1.tabla_reglas(), 1):
    print(f'  R{i:>2}: {r}')
fig = plots.fig_pertenencia(FIS1, ncols=5)
plots.guardar(fig, DIR_NB, 'fis1_pertenencia'); plt.show()
fig = plots.fig_superficies([
    (FIS1, 'retardo', 'carga', 'deseabilidad', {'progreso': 1.0, 'perdida': 0.2}, 'acerca al servidor (pérdida 2 %)'),
    (FIS1, 'retardo', 'perdida', 'deseabilidad', {'progreso': 1.0, 'carga': 0.1}, 'acerca al servidor (carga 10 %)'),
    (FIS1, 'progreso', 'retardo', 'deseabilidad', {'perdida': 0.2, 'carga': 0.1}, 'efecto del progreso')])
plots.guardar(fig, DIR_NB, 'fis1_superficies'); plt.show()

**Lectura:** moverse hacia el servidor por un enlace de bajo retardo, baja pérdida y poca carga es "muy deseable" (≈ 0.9); alejarse es "muy poco deseable" pero no imposible. Así la caminata aleatoria se concentra en rutas QoS-prometedoras **sin perder diversidad**.

La guía se recalcula en cada intervalo con el estado medido (la carga $c_e$ cambia con β), así que **se adapta a la dinámica de la red sin entrenamiento**. Visualización sobre Oxford (flechas más oscuras/gruesas = movimiento más deseable hacia el servidor ★):

In [ ]:
g_ox, _ = cargar_topologia('Oxford')
red_demo = RedSDN(g_ox, n_servidores=2, beta=0.2, semilla=2025)
guia_demo = construir_guia(red_demo, red_demo.servidores)
pos_ox = nx.kamada_kawai_layout(g_ox)
fig = plots.fig_guia(red_demo, guia_demo, red_demo.servidores[0], pos_ox,
                     'FIS-1 en Oxford: deseabilidad de cada movimiento hacia el servidor ★')
plots.guardar(fig, DIR_NB, 'fis1_guia_oxford'); plt.show()

# ¿Qué cambia en las rutas iniciales? 2 000 caminatas RSBI uniformes vs guiadas para la misma solicitud
from src.gamr_original import rsbi
from src.proposed_method import rsbi_guiado
h, s = red_demo.hosts[5], red_demo.servidores[0]
ev = Evaluador(red_demo, [(h, s, 1.0)])
rnd = random.Random(0)
uni = [ev.evaluar([rsbi(red_demo, h, s, rnd)])[0] for _ in range(2000)]
gui = [ev.evaluar([rsbi_guiado(red_demo, h, s, rnd, guia_demo)])[0] for _ in range(2000)]
comp = pd.DataFrame({'RSBI uniforme (GAMR)': np.mean(uni, axis=0), 'RSBI guiada (FIS-1)': np.mean(gui, axis=0)},
                    index=['retardo medio [ms]', 'pérdida media', 'f3 (utilización cuello de botella)']).T
comp['rutas distintas (de 2000)'] = [len({tuple(np.round(x, 6)) for x in uni}), len({tuple(np.round(x, 6)) for x in gui})]
display(comp.round(3))

### 5.3 FIS-2: selector difuso de compromiso del frente
GAMR instala la solución del frente con **menor solapamiento de enlaces**, criterio topológico que no mira la QoS del usuario. FIS-2 recibe, para cada solución del frente $\mathcal{F}^1$, el retardo y la pérdida medios **normalizados min–max dentro del frente** y entrega una aptitud; se instala
$$\Phi^\star=\arg\max_{\Phi\in\mathcal{F}^1}\ \text{FIS-2}\big(\hat{\mathcal{D}}(\Phi),\hat{\mathcal{L}}(\Phi)\big)\quad(\text{empate}\to\text{menor retardo}).$$
Las 6 reglas expresan un compromiso: "ambos bajos" es lo mejor; "uno bajo y el otro medio" es bueno; cualquiera "alto" es malo. La normalización dentro del frente hace que el selector sea **invariante a la escala** de cada instancia.

In [ ]:
print(f'FIS-2: {len(FIS2.reglas)} reglas')
for i, r in enumerate(FIS2.tabla_reglas(), 1):
    print(f'  R{i}: {r}')
fig = plots.fig_pertenencia(FIS2, ncols=3)
plots.guardar(fig, DIR_NB, 'fis2_pertenencia'); plt.show()
fig = plots.fig_superficies([(FIS2, 'delay', 'perdida', 'aptitud', {}, 'superficie de decisión de FIS-2')])
plots.guardar(fig, DIR_NB, 'fis2_superficie'); plt.show()

### 5.4 Costo computacional
* **FIS-1:** se evalúa una vez por intervalo para todos los movimientos $(u,v,S)$: $\mathcal{O}(N_s\,|\mathcal{E}|\,R\,U)$ con $R=10$ reglas y $U=201$ puntos del universo (vectorizado). Dentro del AG, la ruleta cuesta lo mismo que la elección uniforme: $\mathcal{O}(\deg(u))$ por paso.
* **FIS-2:** una evaluación sobre el frente final, $\mathcal{O}(|\mathcal{F}^1|\,R\,U)$.
* La complejidad de GAMR, $\mathcal{O}\big(G^{\max}(NN_r|\mathcal{V}|^2+|\mathcal{E}|+4mN^2)\big)$, **no cambia**; el tiempo real incluso baja porque la guía converge en menos generaciones (Sección 10).

## 6. Demostración en una instancia (Oxford, Nₛ = 2, Nᵣ = 50, β = 20 %)

Se ejecutan GAMR y F-GAMR sobre **la misma red y las mismas solicitudes** durante dos intervalos (en el primero se llena la memoria PRSM). Se muestran la convergencia del hipervolumen con una escala común, el frente obtenido con la solución que instalaría cada criterio y la QoS medida en el estado realizado Δᵗ⁺¹.

In [ ]:
red_i = RedSDN(g_ox, n_servidores=2, beta=0.2, semilla=2025)
algs_i = {'GAMR': GAMR(red_i, semilla=1), 'F-GAMR': FGAMR(red_i, semilla=1)}
for t in (1, 2):
    sol_i = red_i.generar_solicitudes(50, 2)
    r_g = algs_i['GAMR'].resolver(sol_i)
    r_f = algs_i['F-GAMR'].resolver(sol_i, escala=r_g['escala'])
    if t == 1:
        red_i.nuevo_intervalo()
res_i = {'GAMR': r_g, 'F-GAMR': r_f}
fig = plots.fig_convergencia(res_i, 'Convergencia del HV (intervalo 2, escala común)')
plots.guardar(fig, DIR_NB, 'demo_convergencia'); plt.show()

hv_igd = metricas_conjuntas({k: r['frente'] for k, r in res_i.items()})
# Criterios de elección aplicados al MISMO frente de F-GAMR
ind = r_f['individuos_frente']
att = atributos_frente(red_i, r_f['evaluador'], ind)
F = np.column_stack([att['delay'], att['perdida']])
rango = np.ptp(F, axis=0)
elegidos = {'Menor solapamiento (GAMR)': int(np.lexsort((att['delay'], att['solapamiento']))[0]),
            'Mínimo delay': int(np.argmin(att['delay'])),
            'Punto rodilla': int(np.argmin(np.linalg.norm((F - F.min(0)) / np.where(rango > 0, rango, 1), axis=1))),
            'FIS-2': int(np.lexsort((att['delay'], -np.round(aptitud_difusa(att), 9)))[0])}
fig = plots.fig_frente_seleccion(F, elegidos, 'Frente de F-GAMR y la solución que instala cada criterio')
plots.guardar(fig, DIR_NB, 'demo_seleccion'); plt.show()

red_i.nuevo_intervalo()                      # la red evoluciona con las rutas instaladas → QoS realizada
filas = []
for k, r in res_i.items():
    filas.append(dict(algoritmo=k, HV=hv_igd[k][0], IGD=hv_igd[k][1], generaciones=r['generaciones'],
                      tiempo_s=r['tiempo'], **evaluar_qos(red_i, sol_i, r['rutas'])))
display(pd.DataFrame(filas).set_index('algoritmo').round(3))

**Lectura:** la curva de F-GAMR parte más arriba (su población inicial ya contiene rutas QoS-prometedoras) y alcanza un HV mayor, normalmente en menos generaciones. En el frente, "menor solapamiento" puede caer en cualquier zona del compromiso, mientras que FIS-2 elige consistentemente una solución equilibrada (retardo y pérdida simultáneamente bajos). Una sola instancia **no es evidencia**: la evidencia estadística está en las Secciones 10–13.

## 7. Calibración en semillas de validación (500–503)

Todos los parámetros de diseño se fijaron **antes** de mirar las semillas de evaluación, con `python scripts/calibracion.py` (Oxford y Chinanet, β ∈ {5 %, 55 %}, Nₛ = 2, Nᵣ = 50, 4 semillas × 3 intervalos con memoria).

In [ ]:
DIR_CAL = RAIZ / 'results' / 'calibracion'
cols = ['HV', 'IGD', 'tiempo', 'gen', 'alpha', 'delay_ms', 'loss_pct']
for nombre, titulo in [('etapaA_controlador_adaptativo', 'Etapa A — controlador difuso de α / inmigrantes (DESCARTADO)'),
                       ('etapaB_kappa', 'Etapa B — exploración guiada por FIS-1, κ ∈ {1, 2, 4} (κ = 2 ADOPTADO)')]:
    d = pd.read_csv(DIR_CAL / f'{nombre}.csv')
    display(Markdown(f'**{titulo}** (media de t ≥ 2)'))
    display(d[d.t >= 2].groupby(['topo', 'beta', 'alg'])[[c for c in cols if c in d]].mean().round(3))

sel = pd.read_csv(DIR_CAL / 'etapaC_selectores.csv')
g = sel.groupby(['topo', 'beta', 'selector'])[['delay_ms', 'loss_pct']].mean()
ref = g.xs('Menor solapamiento (GAMR)', level='selector')
rel = (100 * (g / ref.reindex(g.index.droplevel('selector')).values - 1)).round(2)
display(Markdown('**Etapa C — bases de reglas de FIS-2: Δ% de la QoS realizada vs "menor solapamiento"** (negativo = mejor)'))
display(rel.unstack('selector'))

**Conclusiones de la calibración**
* **Etapa A:** adaptar α/inmigrantes según la volatilidad no mejora el HV de forma consistente y aumenta el tiempo; con α alto (0.8) en red estable el HV cae hasta 19 %. → descartado (se discute en la Sección 15).
* **Etapa B:** la guía difusa mejora el HV en los 4 escenarios; κ = 2 ofrece el mejor equilibrio (κ = 4 es demasiado codicioso: menos diversidad).
* **Etapa C:** la base retardo–pérdida (B) reduce la pérdida en los 4 escenarios con un costo de retardo acotado; agregar solapamiento/holgura/volatilidad no ayuda.

## 8. Protocolo experimental

| | E1 — comparación principal + ablación | E2 — escalabilidad | E3 — desorden de red |
|---|---|---|---|
| Topología | Oxford y Chinanet | Oxford | Chinanet |
| Servidores Nₛ | 2 | 2 | 2 |
| Solicitudes Nᵣ | 50 | 10, 20, 50, 100 | 100 |
| Desorden β (Ec. 14) | 5 %, 55 % | 20 % | 0, 5, 20, 55 % |
| Algoritmos | MH, QoS-Dijkstra, NSGA-II, GAMR, solo FIS-1, solo FIS-2, **F-GAMR** | MH, QoS-Dijkstra, GAMR, **F-GAMR** | MH, LCP-WAC*, GAMR, **F-GAMR** |
| Semillas independientes | **30** (0–29) | 10 | 10 |

* **Secuencia dinámica por semilla:** T = 4 intervalos. En cada uno el monitoreo mide Δᵗ, llegan Nᵣ solicitudes nuevas y **todos los algoritmos resuelven exactamente la misma instancia**. El intervalo 1 (memoria vacía) se descarta y las métricas se promedian sobre t = 2…4 → **una observación por semilla**.
* **QoS realizada:** las rutas se instalan y la red evoluciona a Δᵗ⁺¹ (Ec. 14 con el β del escenario). Retardo, pérdida, utilización máxima y satisfacción de ancho de banda se miden **en ese estado realizado**, no en el que se optimizó.
* **HV/IGD:** frentes de los evolutivos de cada instancia normalizados en conjunto (ideal/nadir de la unión), $r^{\text{ref}}=(1,1,1)$, $\mathcal{F}^*$ = no dominados de la unión (Ec. 16–17).
* **Estadística:** media ± desviación estándar sobre semillas; **Wilcoxon de rangos con signo pareado** por semilla (α = 0.05).
* **Parámetros (Tabla III):** N = 100, Gᵐᵃˣ = 500, Gᵖʳᵉ = 10, p_c = 0.8, p_m = 0.1, α = 0.5, torneo binario.
* **Líneas base:** Min-Hop (camino más corto en saltos), QoS-Dijkstra (mínimo retardo sobre enlaces con residual ≥ Bₖ), NSGA-II (α = 0, sin memoria), GAMR original y LCP-WAC* (proxy sin LSTM, solo en E3).

## 9. Ejecución (o carga) de los experimentos

In [ ]:
tareas = configuraciones('completo' if MODO == 'completo' else 'rapido')
print(pd.Series([t['experimento'] for t in tareas]).value_counts().sort_index().to_string(), '\n')
t0 = time.time()
df, df_sel = cargar_o_ejecutar(DIR_CRUDOS, MODO, N_JOBS, RECALCULAR)
out = generar_reporte(df, df_sel, DIR_TABLAS, DIR_FIGS)
print(f'{len(df)} filas · {df.semilla.nunique()} semillas · {time.time() - t0:.0f} s')


def tabla_delta(t, filtro=None, col_grupo='esc'):
    """Pivote 'Δ% (> 0 = mejor) con * si p < 0.05' a partir de una tabla de comparar()."""
    t = t if filtro is None else t[filtro(t)]
    celda = t.apply(lambda r: f"{r.mejora_pct:+.1f}%{'*' if r.p_valor < 0.05 else ''}", axis=1)
    return t.assign(celda=celda).pivot_table(index=[col_grupo, 'algoritmo'], columns='metrica', values='celda', aggfunc='first')

## 10. E1 — Calidad del frente de Pareto y tiempo (30 semillas)

In [ ]:
display(Markdown('**Media ± std por semilla (evolutivos)**'))
display(out['E1_moo'])
display(Image(filename=str(DIR_FIGS / 'E1_cajas_moo.png'), width=1000))
display(Markdown('**Ablación: Δ% frente a GAMR original** (> 0 = mejor; * = Wilcoxon p < 0.05)'))
display(tabla_delta(out['E1_ablacion'], lambda t: t.metrica.isin(['HV', 'IGD', 'tiempo_s']))) 

## 11. E1 — QoS realizada y análisis de la selección final

In [ ]:
display(Markdown('**Media ± std por semilla (todos los algoritmos)**'))
display(out['E1_qos'])
display(Image(filename=str(DIR_FIGS / 'E1_barras_qos.png'), width=1000))
display(Markdown('**F-GAMR frente a las tres líneas base** (Δ%, > 0 = mejor; * = p < 0.05)'))
m = out['E1_mejoras'].copy()
m['algoritmo'] = 'F-GAMR vs ' + m['referencia']
display(tabla_delta(m))
display(Markdown('**Ablación de la QoS frente a GAMR** (Δ%, * = p < 0.05)'))
display(tabla_delta(out['E1_ablacion'], lambda t: t.metrica.isin(['delay_ms', 'loss_pct'])))

**Selección final aislada.** Sobre **los mismos frentes de GAMR** (sin alterar su memoria) se compara la QoS realizada de cuatro criterios de elección: el de GAMR (menor solapamiento), mínimo retardo, punto rodilla y FIS-2. Así se mide el aporte de FIS-2 sin la influencia de FIS-1.

In [ ]:
if 'selectores' in out:
    display(out['selectores'])
    display(Markdown('**Δ% de cada criterio frente a "menor solapamiento"** (> 0 = mejor; * = p < 0.05)'))
    display(tabla_delta(out['selectores_wilcoxon']))

## 12. E2 — Escalabilidad en el número de solicitudes (Oxford, Nₛ = 2, β = 20 %, 10 semillas)

In [ ]:
if 'E2' in out:
    display(out['E2'])
    display(Image(filename=str(DIR_FIGS / 'E2_escalabilidad.png'), width=950))
    display(Markdown('**Δ% frente a GAMR** (* = p < 0.05)'))
    display(tabla_delta(out['E2_wilcoxon'], col_grupo='n_r'))

## 13. E3 — Robustez ante el desorden del ancho de banda β (Chinanet, Nₛ = 2, Nᵣ = 100, 10 semillas)

In [ ]:
if 'E3' in out:
    display(out['E3'])
    display(Image(filename=str(DIR_FIGS / 'E3_beta.png'), width=950))
    display(Markdown('**Δ% frente a GAMR** (* = p < 0.05)'))
    display(tabla_delta(out['E3_wilcoxon'], col_grupo='beta_pct'))

## 14. Resumen numérico automático
Archivo `results/tables/resumen_resultados.md` (insumo directo para el informe y la presentación).

In [ ]:
display(Markdown((DIR_TABLAS / 'resumen_resultados.md').read_text(encoding='utf-8')))

## 15. Discusión crítica, casos de fallo y limitaciones

<!-- DISCUSION -->
*(Esta sección se completa con los resultados del modo completo.)*

## 16. Conclusiones y trabajo futuro

<!-- CONCLUSIONES -->

## 17. Reproducibilidad

| Paso | Comando |
|---|---|
| Instalar | `python -m venv .venv` → activar → `pip install -r requirements.txt` |
| Pruebas | `python -m pytest -q` (7 pruebas, < 10 s) |
| Experimentos + tablas + figuras | `python run_reproduction.py` (≈ 1.5 h, 7 procesos) · `--modo rapido` (≈ 5 min) |
| Solo regenerar tablas/figuras | `python run_reproduction.py --solo-reporte` |
| Calibración (semillas 500–503) | `python scripts/calibracion.py` (≈ 15 min) |
| Este notebook | Run All con `MODO = 'completo'` (carga `results/raw_logs/`) |

* **Determinismo:** cada tarea (experimento, semilla) fija la semilla de la red (`10 000 + 97·s`) y la de cada algoritmo (`1000·s + k`); los algoritmos no consumen el generador de la red, así que todos ven las mismas solicitudes y estados. Repetir la corrida reproduce los CSV exactamente (salvo los tiempos de CPU).
* **Hardware de referencia:** Intel Core i5-10210U (4 núcleos / 8 hilos), Windows 11, Python 3.12.3. Los tiempos absolutos dependen del equipo; las comparaciones de tiempo son válidas porque todos los algoritmos de una tarea corren en el mismo proceso, uno tras otro.
* **Sin dependencias frágiles:** solo numpy, scipy, pandas, networkx y matplotlib; la lógica difusa es propia (`src/fuzzy_logic.py`).